# Granite-IO Interactive Chat Agent with Ollama

Welcome to this hands-on tutorial on **Granite-IO**, a powerful Python framework that enables you to transform how users interact with IBM Granite models and customize the input/output processing.

## What is Granite-IO?

Granite-IO is a framework that lets you:
- 🔄 Transform how users call or infer IBM Granite models
- 🎯 Customize how outputs are returned to users
- ⚡ Extend model functionality through input/output processing
- 🧠 Enable advanced features like "thinking" mode
- 📚 Build RAG (Retrieval-Augmented Generation) applications

## What We'll Build

In this notebook, we'll create an interactive chat agent that:
1. Uses **Ollama** as the backend to run Granite models locally
2. Provides a simple chat interface where you can ask questions
3. Demonstrates both regular and "thinking" modes
4. Shows how easy it is to get started with Granite-IO
5. Uses the latest **Granite 3.3** models with enhanced capabilities

## Why Granite 3.3?

Granite 3.3 models offer significant improvements over previous versions:
- **🎯 Better Performance**: Enhanced reasoning and instruction-following
- **🧠 Advanced Thinking**: Improved structured reasoning capabilities
- **📈 Benchmark Gains**: Better scores on AlpacaEval-2.0 and Arena-Hard
- **🔧 More Capabilities**: Enhanced coding, mathematics, and multilingual support
- **📝 Fill-in-the-Middle**: Support for code completion tasks
- **🌍 More Languages**: Support for 12+ languages including English, German, Spanish, French, Japanese, and more

## Prerequisites

- Python 3.10+
- Ollama installed and running locally
- Granite 3.3 model available in Ollama

## Step 1: Installation and Setup

First, let's install the required packages and set up our environment.

In [ ]:
# Install granite-io with OpenAI compatibility (for Ollama)
!pip install granite-io[openai] -q

print("✅ Granite-IO installed successfully!")
print("📦 Next, make sure you have:")
print("   1. Ollama installed and running")
print("   2. Granite model available (run: ollama pull granite3.3:8b)")
print("   3. Ollama server running on localhost:11434")

In [ ]:
# Import required libraries
import requests
import json
from granite_io import make_backend, make_io_processor
from granite_io.types import ChatCompletionInputs, UserMessage

# Test Ollama connectivity
try:
    response = requests.get("http://localhost:11434/api/tags")
    if response.status_code == 200:
        models = response.json()
        print("✅ Ollama is running and accessible!")
        print("📋 Available models:")
        for model in models.get("models", []):
            print(f"   - {model['name']}")
    else:
        print("❌ Ollama is not accessible. Please start Ollama first.")
except Exception as e:
    print(f"❌ Cannot connect to Ollama: {e}")
    print("💡 Make sure Ollama is installed and running (try: ollama serve)")

## Step 2: Initialize Granite-IO with Ollama Backend

Now let's create our Granite-IO processor that will interface with Ollama. This is where the magic happens!

In [ ]:
# Configure the model and backend
model_name = "granite3.3:8b"

# Create the backend configuration for Ollama
# Ollama uses OpenAI-compatible API, so we use the "openai" backend
backend_config = {
    "model_name": model_name,
    "base_url": "http://localhost:11434/v1",  # Ollama's OpenAI-compatible endpoint
    "api_key": "ollama"  # Ollama doesn't require a real API key, but the client expects one
}

try:
    # Create the backend and IO processor
    backend = make_backend("openai", backend_config)
    io_processor = make_io_processor(model_name, backend=backend)
    
    print("🎉 Granite-IO processor created successfully!")
    print(f"🤖 Using model: {model_name}")
    print("🔌 Backend: Ollama (via OpenAI-compatible API)")
    
except Exception as e:
    print(f"❌ Error creating processor: {e}")
    print("💡 Make sure Ollama is running and the granite3.3:8b model is available")

## Step 3: Your First Chat with Granite-IO

Let's test our setup with a simple question!

In [ ]:
# Let's try a simple question
user_question = "What is the capital of France?"

# Create a user message
messages = [UserMessage(content=user_question)]

# Create chat completion input
chat_input = ChatCompletionInputs(messages=messages)

try:
    # Get response from Granite via Ollama
    print(f"🗣️  You: {user_question}")
    print("🤖 Granite: ", end="")
    
    outputs = io_processor.create_chat_completion(chat_input)
    response = outputs.results[0].next_message.content
    
    print(response)
    print("\n✅ Success! Your Granite-IO chat agent is working!")
    
except Exception as e:
    print(f"❌ Error: {e}")
    print("💡 Check that Ollama is running and the model is available")

## Step 4: Exploring the "Thinking" Feature

One of Granite-IO's powerful features is the ability to see the model's reasoning process. Let's try the same question with "thinking" enabled!

In [ ]:
# Let's ask a more complex question that benefits from reasoning
thinking_question = "What's the best strategy for a traveling salesperson to visit 5 cities efficiently?"

# Create messages for thinking mode
messages = [UserMessage(content=thinking_question)]

try:
    print(f"🗣️  You: {thinking_question}")
    print("\n" + "="*60)
    
    # First, let's see the response WITHOUT thinking
    print("🤖 WITHOUT THINKING:")
    print("-" * 20)
    regular_outputs = io_processor.create_chat_completion(ChatCompletionInputs(messages=messages))
    regular_response = regular_outputs.results[0].next_message.content
    print(regular_response)
    
    print("\n" + "="*60)
    
    # Now WITH thinking enabled
    print("🧠 WITH THINKING:")
    print("-" * 20)
    thinking_outputs = io_processor.create_chat_completion(
        ChatCompletionInputs(messages=messages, thinking=True)
    )
    
    # Show the reasoning process
    print("💭 Model's Thoughts:")
    reasoning = thinking_outputs.results[0].next_message.reasoning_content
    if reasoning:
        print(reasoning)
    else:
        print("(No reasoning content available)")
    
    print("\n🎯 Final Response:")
    thinking_response = thinking_outputs.results[0].next_message.content
    print(thinking_response)
    
except Exception as e:
    print(f"❌ Error: {e}")

## Step 5: Interactive Chat Agent

Now let's create a full interactive chat experience! Run the cell below and start chatting with your Granite model.

In [ ]:
def interactive_chat():
    """
    Interactive chat function with conversation history
    """
    print("🎯 Interactive Granite Chat Agent")
    print("=" * 50)
    print("💬 Type your questions (or 'quit' to exit)")
    print("🧠 Add 'think:' at the start to enable thinking mode")
    print("🔄 Type 'clear' to clear conversation history")
    print("=" * 50)
    
    # Conversation history
    conversation_history = []
    
    while True:
        try:
            # Get user input
            user_input = input("\n🗣️  You: ").strip()
            
            if user_input.lower() == 'quit':
                print("👋 Goodbye! Thanks for chatting!")
                break
            elif user_input.lower() == 'clear':
                conversation_history = []
                print("🧹 Conversation history cleared!")
                continue
            elif not user_input:
                continue
            
            # Check if thinking mode is requested
            thinking_mode = user_input.lower().startswith('think:')
            if thinking_mode:
                user_input = user_input[6:].strip()  # Remove 'think:' prefix
            
            # Add user message to history
            conversation_history.append(UserMessage(content=user_input))
            
            # Create chat input with full conversation history
            chat_input = ChatCompletionInputs(
                messages=conversation_history,
                thinking=thinking_mode
            )
            
            # Get response
            print("🤖 Granite: ", end="")
            if thinking_mode:
                print("(thinking...)")
            
            outputs = io_processor.create_chat_completion(chat_input)
            result = outputs.results[0].next_message
            
            # Show thinking if available
            if thinking_mode and result.reasoning_content:
                print(f"💭 Thoughts: {result.reasoning_content}")
                print(f"🎯 Response: {result.content}")
            else:
                print(result.content)
            
            # Add assistant response to history (simplified - just the content)
            from granite_io.types import AssistantMessage
            conversation_history.append(AssistantMessage(content=result.content))
            
        except KeyboardInterrupt:
            print("\n👋 Chat interrupted. Goodbye!")
            break
        except Exception as e:
            print(f"\n❌ Error: {e}")
            print("💡 Try again or check your Ollama setup")

# Start the interactive chat
interactive_chat()

## 🎉 Congratulations!

You've successfully created an interactive chat agent using **Granite-IO** with **Ollama**! 

### What You've Learned:

1. **🔧 Setup**: How to install and configure Granite-IO with Ollama
2. **🔌 Backend**: Using Ollama as a local backend for Granite models
3. **💬 Basic Chat**: Creating simple chat completions
4. **🧠 Thinking Mode**: Accessing the model's reasoning process
5. **🔄 Interactive Chat**: Building a full conversational agent with history

### Key Granite-IO Concepts:

- **`make_backend()`**: Creates the connection to your model backend (Ollama, vLLM, etc.)
- **`make_io_processor()`**: The main interface for processing requests and responses
- **`ChatCompletionInputs`**: Structure your chat inputs with messages and options
- **`UserMessage`/`AssistantMessage`**: Type-safe message handling
- **`thinking=True`**: Enable reasoning mode to see model's thought process

### Next Steps:

- Try different Granite models (granite3.3:2b, granite3.3:8b)
- Experiment with system messages and custom prompts
- Explore RAG (Retrieval-Augmented Generation) capabilities
- Build custom input/output processors

### 🔗 Learn More:

- [Granite-IO Documentation](https://github.com/ibm-granite/granite-io)
- [Ollama Documentation](https://ollama.readthedocs.io/)
- [Granite Models on Hugging Face](https://huggingface.co/ibm-granite)